# Maxime — participants, fiabilité et calendrier 2026-2027

Ce notebook prépare les **36 participants** à la phase de ligue, mesure la quantité
d’historique disponible et reconstitue les **144 rencontres UEFA**. Il produit :

- `output/participants_2026.csv` : une ligne par club, correspondance des noms et diagnostic de fiabilité ;
- `output/calendrier_2026.csv` : **288 lignes**, une ligne par club et par journée, avec son adversaire et le lieu.

Les dictionnaires sont inclus dans le code : aucun CSV de participants ou de calendrier
n’est nécessaire en entrée. Ils sont donc partageables via Git malgré le `.gitignore`.
Exécuter toutes les cellules, depuis la racine du dépôt ou depuis `analyse/`.
Seule dépendance de données : `data/games.csv` ; dépendance Python : `pandas` et Jupyter.

**Périmètre de prévision : avant le début de la phase de ligue, le 8 septembre 2026.**
Les scores 2026-2027 présents sur les pages UEFA ne sont pas importés. La fiabilité
désigne la solidité de l’historique dans notre fichier, pas la force sportive du club.

## 1. Sources et conventions

Sources UEFA consultées le **19 octobre 2026** :

1. [Présentation des 36 équipes et modes de qualification](https://fr.uefa.com/uefachampionsleague/news/02a8-2172206d3905-b8c725f80fa3-1000--presentation-des-equipes-de-la-phase-de-ligue-de-la-ligue-d/).
2. [Tirage : quatre adversaires à domicile et quatre à l’extérieur](https://es.uefa.com/uefachampionsleague/news/02a8-21731fe7fcd9-f7fc3f29d628-1000--sorteo-de-la-fase-liga-de-la-champions-league-los-36-equi/).
3. [Calendrier UEFA par journée et par date](https://www.uefa.com/uefachampionsleague/news/02a8-2174c9e9019d-f909a77bd77a-1000--2026-27-champions-league-all-the-league-phase-fixture/).

Les noms UEFA sont ceux de la liste des participants (par exemple `Paris`, `Man City`
et `FC Barcelona`). Les variantes des pages de calendrier sont harmonisées dans le
dictionnaire. Les identifiants et noms `nom_games` proviennent des colonnes domicile
et extérieur de `games.csv` : **les jointures utilisent `club_id`**.
Sabah est identifié dans `games.csv`, même s’il est absent de `clubs.csv`.

`competition_id == "CL"` désigne la compétition principale ; `CLQ` désigne les qualifications,
comptées séparément. `season` est l’année de début : 2025 signifie 2025-2026.
Le mode de qualification décrit l’accès à la phase de ligue selon la fiche UEFA :
un vainqueur des barrages reste étiqueté « barrages », même s’il est champion national.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

SAISON_CIBLE = 2026
DATE_DEBUT_PHASE_LIGUE = pd.Timestamp("2026-09-08")
DATE_VERIFICATION_UEFA = "2026-10-19"
SOURCE_PARTICIPANTS = "https://fr.uefa.com/uefachampionsleague/news/02a8-2172206d3905-b8c725f80fa3-1000--presentation-des-equipes-de-la-phase-de-ligue-de-la-ligue-d/"
SOURCE_TIRAGE = "https://es.uefa.com/uefachampionsleague/news/02a8-21731fe7fcd9-f7fc3f29d628-1000--sorteo-de-la-fase-liga-de-la-champions-league-los-36-equi/"
SOURCE_CALENDRIER = "https://www.uefa.com/uefachampionsleague/news/02a8-2174c9e9019d-f909a77bd77a-1000--2026-27-champions-league-all-the-league-phase-fixture/"

racines = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in racines if (p / "data/games.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Placer games.csv dans data/, puis exécuter depuis le dépôt ou analyse/.")
OUTPUT = ROOT / "output"

colonnes = ["game_id", "competition_id", "season", "date", "home_club_id", "away_club_id",
            "home_club_name", "away_club_name"]
games_brut = pd.read_csv(ROOT / "data/games.csv", usecols=colonnes)
assert games_brut[colonnes[:6]].notna().all().all(), "Identifiant, compétition, saison ou date manquant."
games_brut["date"] = pd.to_datetime(games_brut["date"], errors="raise")
games_brut = games_brut.drop_duplicates()
assert games_brut["game_id"].is_unique, "Plusieurs lignes différentes pour un même game_id."
assert games_brut["home_club_id"].ne(games_brut["away_club_id"]).all()

# Cette double borne exclut une éventuelle mise à jour avec des matchs de 2026-2027.
games = games_brut.loc[
    games_brut["season"].lt(SAISON_CIBLE)
    & games_brut["date"].lt(DATE_DEBUT_PHASE_LIGUE)
].copy()
assert not games.empty, "Aucun match historique exploitable."
ucl = games.loc[games["competition_id"].eq("CL")]
assert not ucl.empty, "La compétition CL n’est pas présente dans games.csv."
DERNIERE_SAISON_REFERENCE = SAISON_CIBLE - 1
PREMIERE_SAISON_RECENTE = SAISON_CIBLE - 3
print(f"{len(games):,} matchs avant la phase de ligue ; {len(ucl):,} matchs de C1.")
print(f"C1 observée : saisons {ucl.season.min()} à {ucl.season.max()}, dernier match le {ucl.date.max():%d/%m/%Y}.")

88,958 matchs avant la phase de ligue ; 1,871 matchs de C1.
C1 observée : saisons 2012 à 2025, dernier match le 06/05/2026.


## 2. Les 36 participants : dictionnaire partagé

Cette cellule constitue la liste de référence versionnée. Une vérification ci-dessous
contrôle chaque couple identifiant/nom contre le fichier local et signale toute dérive.

In [2]:
PARTICIPANTS_2026 = {'AEK Athens': {'nom_games': 'Athlitiki Enosi Konstantinoupoleos',
                'club_id': 2441,
                'pays': 'Grèce',
                'code_pays': 'GRE',
                'voie_qualification': 'barrages',
                'mode_qualification': 'Vainqueur des barrages'},
 'Arsenal': {'nom_games': 'Arsenal FC',
             'club_id': 11,
             'pays': 'Angleterre',
             'code_pays': 'ENG',
             'voie_qualification': 'championnat',
             'mode_qualification': '1er en Angleterre'},
 'Aston Villa': {'nom_games': 'Aston Villa',
                 'club_id': 405,
                 'pays': 'Angleterre',
                 'code_pays': 'ENG',
                 'voie_qualification': 'tenant_europa_league',
                 'mode_qualification': 'Vainqueur de l’Europa League'},
 'Atlético de Madrid': {'nom_games': 'Atlético de Madrid',
                        'club_id': 13,
                        'pays': 'Espagne',
                        'code_pays': 'ESP',
                        'voie_qualification': 'championnat',
                        'mode_qualification': '4e en Espagne'},
 'FC Barcelona': {'nom_games': 'FC Barcelona',
                  'club_id': 131,
                  'pays': 'Espagne',
                  'code_pays': 'ESP',
                  'voie_qualification': 'championnat',
                  'mode_qualification': '1er en Espagne'},
 'Bayern München': {'nom_games': 'Bayern Munich',
                    'club_id': 27,
                    'pays': 'Allemagne',
                    'code_pays': 'GER',
                    'voie_qualification': 'championnat',
                    'mode_qualification': '1er en Allemagne'},
 'Bodø/Glimt': {'nom_games': 'Fotballklubben Bodø/Glimt',
                'club_id': 501,
                'pays': 'Norvège',
                'code_pays': 'NOR',
                'voie_qualification': 'barrages',
                'mode_qualification': 'Vainqueur des barrages'},
 'Borussia Dortmund': {'nom_games': 'Borussia Dortmund',
                       'club_id': 16,
                       'pays': 'Allemagne',
                       'code_pays': 'GER',
                       'voie_qualification': 'championnat',
                       'mode_qualification': '2e en Allemagne'},
 'Club Brugge': {'nom_games': 'Club Brugge KV',
                 'club_id': 2282,
                 'pays': 'Belgique',
                 'code_pays': 'BEL',
                 'voie_qualification': 'championnat',
                 'mode_qualification': '1er en Belgique'},
 'Como': {'nom_games': 'Como 1907',
          'club_id': 1047,
          'pays': 'Italie',
          'code_pays': 'ITA',
          'voie_qualification': 'championnat',
          'mode_qualification': '4e en Italie'},
 'Fenerbahçe': {'nom_games': 'Fenerbahce',
                'club_id': 36,
                'pays': 'Turquie',
                'code_pays': 'TUR',
                'voie_qualification': 'barrages',
                'mode_qualification': 'Vainqueur des barrages'},
 'Feyenoord': {'nom_games': 'Feyenoord Rotterdam',
               'club_id': 234,
               'pays': 'Pays-Bas',
               'code_pays': 'NED',
               'voie_qualification': 'championnat',
               'mode_qualification': '2e aux Pays-Bas'},
 'Galatasaray': {'nom_games': 'Galatasaray',
                 'club_id': 141,
                 'pays': 'Turquie',
                 'code_pays': 'TUR',
                 'voie_qualification': 'championnat',
                 'mode_qualification': '1er en Turquie'},
 'Inter': {'nom_games': 'Inter Milan',
           'club_id': 46,
           'pays': 'Italie',
           'code_pays': 'ITA',
           'voie_qualification': 'championnat',
           'mode_qualification': '1er en Italie'},
 'LASK': {'nom_games': 'LASK',
          'club_id': 413,
          'pays': 'Autriche',
          'code_pays': 'AUT',
          'voie_qualification': 'barrages',
          'mode_qualification': 'Vainqueur des barrages'},
 'Leipzig': {'nom_games': 'RB Leipzig',
             'club_id': 23826,
             'pays': 'Allemagne',
             'code_pays': 'GER',
             'voie_qualification': 'championnat',
             'mode_qualification': '3e en Allemagne'},
 'Lens': {'nom_games': 'RC Lens',
          'club_id': 826,
          'pays': 'France',
          'code_pays': 'FRA',
          'voie_qualification': 'championnat',
          'mode_qualification': '2e en France'},
 'Lille': {'nom_games': 'LOSC Lille',
           'club_id': 1082,
           'pays': 'France',
           'code_pays': 'FRA',
           'voie_qualification': 'championnat',
           'mode_qualification': '3e en France'},
 'Liverpool': {'nom_games': 'Liverpool FC',
               'club_id': 31,
               'pays': 'Angleterre',
               'code_pays': 'ENG',
               'voie_qualification': 'championnat',
               'mode_qualification': '5e en Angleterre'},
 'Man City': {'nom_games': 'Manchester City',
              'club_id': 281,
              'pays': 'Angleterre',
              'code_pays': 'ENG',
              'voie_qualification': 'championnat',
              'mode_qualification': '2e en Angleterre'},
 'Man United': {'nom_games': 'Manchester United',
                'club_id': 985,
                'pays': 'Angleterre',
                'code_pays': 'ENG',
                'voie_qualification': 'championnat',
                'mode_qualification': '3e en Angleterre'},
 'Napoli': {'nom_games': 'SSC Napoli',
            'club_id': 6195,
            'pays': 'Italie',
            'code_pays': 'ITA',
            'voie_qualification': 'championnat',
            'mode_qualification': '2e en Italie'},
 'Paris': {'nom_games': 'Paris Saint-Germain',
           'club_id': 583,
           'pays': 'France',
           'code_pays': 'FRA',
           'voie_qualification': 'tenant_champions_league',
           'mode_qualification': 'Vainqueur de la C1 et 1er en France'},
 'Porto': {'nom_games': 'FC Porto',
           'club_id': 720,
           'pays': 'Portugal',
           'code_pays': 'POR',
           'voie_qualification': 'championnat',
           'mode_qualification': '1er au Portugal'},
 'PSV': {'nom_games': 'PSV Eindhoven',
         'club_id': 383,
         'pays': 'Pays-Bas',
         'code_pays': 'NED',
         'voie_qualification': 'championnat',
         'mode_qualification': '1er aux Pays-Bas'},
 'Real Betis': {'nom_games': 'Real Betis Balompié',
                'club_id': 150,
                'pays': 'Espagne',
                'code_pays': 'ESP',
                'voie_qualification': 'championnat',
                'mode_qualification': '5e en Espagne'},
 'Real Madrid': {'nom_games': 'Real Madrid',
                 'club_id': 418,
                 'pays': 'Espagne',
                 'code_pays': 'ESP',
                 'voie_qualification': 'championnat',
                 'mode_qualification': '2e en Espagne'},
 'Roma': {'nom_games': 'Associazione Sportiva Roma',
          'club_id': 12,
          'pays': 'Italie',
          'code_pays': 'ITA',
          'voie_qualification': 'championnat',
          'mode_qualification': '3e en Italie'},
 'Sabah': {'nom_games': 'Sabah FK',
           'club_id': 63993,
           'pays': 'Azerbaïdjan',
           'code_pays': 'AZE',
           'voie_qualification': 'barrages',
           'mode_qualification': 'Vainqueur des barrages'},
 'Shakhtar': {'nom_games': 'FC Shakhtar Donetsk',
              'club_id': 660,
              'pays': 'Ukraine',
              'code_pays': 'UKR',
              'voie_qualification': 'reequilibrage',
              'mode_qualification': 'Rééquilibrage de la liste d’accès C1'},
 'Slavia Praha': {'nom_games': 'SK Slavia Prague',
                  'club_id': 62,
                  'pays': 'Tchéquie',
                  'code_pays': 'CZE',
                  'voie_qualification': 'championnat',
                  'mode_qualification': '1er en Tchéquie'},
 'Slovan Bratislava': {'nom_games': 'Slovan Bratislava',
                       'club_id': 540,
                       'pays': 'Slovaquie',
                       'code_pays': 'SVK',
                       'voie_qualification': 'barrages',
                       'mode_qualification': 'Vainqueur des barrages'},
 'Sporting CP': {'nom_games': 'Sporting CP',
                 'club_id': 336,
                 'pays': 'Portugal',
                 'code_pays': 'POR',
                 'voie_qualification': 'championnat',
                 'mode_qualification': '2e au Portugal'},
 'Stuttgart': {'nom_games': 'VfB Stuttgart',
               'club_id': 79,
               'pays': 'Allemagne',
               'code_pays': 'GER',
               'voie_qualification': 'championnat',
               'mode_qualification': '4e en Allemagne'},
 'Viking': {'nom_games': 'Viking Fotballklubb',
            'club_id': 239,
            'pays': 'Norvège',
            'code_pays': 'NOR',
            'voie_qualification': 'barrages',
            'mode_qualification': 'Vainqueur des barrages'},
 'Villarreal': {'nom_games': 'Villarreal CF',
                'club_id': 1050,
                'pays': 'Espagne',
                'code_pays': 'ESP',
                'voie_qualification': 'championnat',
                'mode_qualification': '3e en Espagne'}}

In [3]:
participants = pd.DataFrame.from_dict(PARTICIPANTS_2026, orient="index").rename_axis("nom_uefa").reset_index()
participants.insert(0, "saison", SAISON_CIBLE)
assert len(participants) == 36
assert participants["nom_uefa"].is_unique and participants["club_id"].is_unique

# Table des apparitions : une ligne par club et par match, domicile + extérieur.
apparitions = pd.concat([
    games[["game_id", "competition_id", "season", "date", f"{cote}_club_id", f"{cote}_club_name"]]
    .rename(columns={f"{cote}_club_id": "club_id", f"{cote}_club_name": "nom_games"})
    for cote in ("home", "away")
], ignore_index=True)
assert not apparitions.duplicated(["game_id", "club_id"]).any()
for club in participants.itertuples(index=False):
    noms_observes = set(apparitions.loc[apparitions.club_id.eq(club.club_id), "nom_games"].dropna())
    assert club.nom_games in noms_observes, (club.nom_uefa, club.club_id, noms_observes)
    assert noms_observes == {club.nom_games}, f"Variantes à vérifier pour {club.nom_uefa}: {noms_observes}"

with pd.option_context("display.max_rows", 40, "display.max_colwidth", 55):
    display(participants[["nom_uefa", "nom_games", "club_id", "pays", "mode_qualification"]])

,nom_uefa,nom_games,club_id,pays,mode_qualification
0,AEK Athens,Athlitiki Enosi Konstantinoupoleos,2441,Grèce,Vainqueur des barrages
1,Arsenal,Arsenal FC,11,Angleterre,1er en Angleterre
2,Aston Villa,Aston Villa,405,Angleterre,Vainqueur de l’Europa League
3,Atlético de Madrid,Atlético de Madrid,13,Espagne,4e en Espagne
4,FC Barcelona,FC Barcelona,131,Espagne,1er en Espagne
5,Bayern München,Bayern Munich,27,Allemagne,1er en Allemagne
6,Bodø/Glimt,Fotballklubben Bodø/Glimt,501,Norvège,Vainqueur des barrages
7,Borussia Dortmund,Borussia Dortmund,16,Allemagne,2e en Allemagne
8,Club Brugge,Club Brugge KV,2282,Belgique,1er en Belgique
9,Como,Como 1907,1047,Italie,4e en Italie


## 3. Tableau de fiabilité

Chaque match compte une seule fois par club. Les compteurs portent sur les matchs
réellement présents dans le fichier, toutes compétitions pour `nb_matchs_games`, puis
uniquement `CL` pour `nb_matchs_c1`. Les qualifications sont dans `nb_matchs_clq`.
`derniere_saison_c1` reste vide lorsqu’aucun match n’est observé ; on n’invente pas une saison.

Règle descriptive choisie pour le projet, **à valider lors des tests chronologiques du modèle** :

| Fiabilité de l’historique C1 | Règle |
|---|---|
| Très faible | Aucun match de C1 dans `games.csv` |
| Faible | Moins de 10 matchs de C1, ou dernière saison de C1 antérieure à 2023-2024, ou moins de 30 matchs toutes compétitions |
| Modérée | De 10 à 29 matchs de C1 et historique récent |
| Élevée | Au moins 30 matchs de C1 et historique récent |

Ces seuils sont des **hypothèses de travail**, pas des probabilités ni un classement sportif.
La récence se mesure par rapport à la dernière saison avant la cible (2025-2026),
et non par rapport à la date système. La fenêtre récente couvre 2023-2024 à 2025-2026.

In [4]:
stats = apparitions.groupby("club_id").agg(nb_matchs_games=("game_id", "nunique"))
app_c1 = apparitions.loc[apparitions.competition_id.eq("CL")]
stats_c1 = app_c1.groupby("club_id").agg(
    nb_matchs_c1=("game_id", "nunique"),
    derniere_saison_c1_annee=("season", "max"),
)
stats_clq = apparitions.loc[apparitions.competition_id.eq("CLQ")].groupby("club_id").game_id.nunique()
stats_recent = app_c1.loc[app_c1.season.ge(PREMIERE_SAISON_RECENTE)].groupby("club_id").game_id.nunique()
participants = (participants.merge(stats, on="club_id", how="left", validate="one_to_one")
               .merge(stats_c1, on="club_id", how="left", validate="one_to_one"))
participants["nb_matchs_clq"] = participants.club_id.map(stats_clq)
participants["nb_matchs_c1_recents"] = participants.club_id.map(stats_recent)
compteurs = ["nb_matchs_games", "nb_matchs_c1", "nb_matchs_clq", "nb_matchs_c1_recents"]
participants[compteurs] = participants[compteurs].fillna(0).astype(int)
participants["derniere_saison_c1_annee"] = participants.derniere_saison_c1_annee.astype("Int64")
participants["derniere_saison_c1"] = participants.derniere_saison_c1_annee.map(
    lambda s: f"{int(s)}-{int(s) + 1}" if pd.notna(s) else pd.NA
).astype("string")
participants["saisons_depuis_c1"] = DERNIERE_SAISON_REFERENCE - participants.derniere_saison_c1_annee
participants["sans_historique_c1"] = participants.nb_matchs_c1.eq(0)

def diagnostic_fiabilite(club):
    raisons = []
    if club.nb_matchs_c1 == 0:
        raisons.append("Aucun match CL dans le fichier")
    elif club.nb_matchs_c1 < 10:
        raisons.append("Moins de 10 matchs de C1")
    elif club.nb_matchs_c1 < 30:
        raisons.append("Moins de 30 matchs de C1")
    if pd.notna(club.saisons_depuis_c1) and club.saisons_depuis_c1 >= 3:
        raisons.append(f"Historique C1 ancien : dernière saison {club.derniere_saison_c1}")
    if club.nb_matchs_games < 30:
        raisons.append("Moins de 30 matchs toutes compétitions : couverture très limitée")
    if club.sans_historique_c1:
        niveau = "Très faible"
    elif (club.nb_matchs_c1 < 10 or club.saisons_depuis_c1 >= 3 or club.nb_matchs_games < 30):
        niveau = "Faible"
    elif club.nb_matchs_c1 < 30:
        niveau = "Modérée"
    else:
        niveau = "Élevée"
    return niveau, "; ".join(raisons) or "Au moins 30 matchs de C1 et présence depuis 2023-2024"

diagnostics = [diagnostic_fiabilite(c) for c in participants.itertuples(index=False)]
participants["fiabilite"] = [d[0] for d in diagnostics]
participants["raison_fiabilite"] = [d[1] for d in diagnostics]
with pd.option_context("display.max_rows", 40, "display.max_colwidth", 100):
    display(participants[["nom_uefa", "nb_matchs_games", "nb_matchs_c1", "nb_matchs_clq",
                          "derniere_saison_c1", "nb_matchs_c1_recents", "fiabilite", "raison_fiabilite"]])

,nom_uefa,nb_matchs_games,nb_matchs_c1,nb_matchs_clq,derniere_saison_c1,nb_matchs_c1_recents,fiabilite,raison_fiabilite
0,AEK Athens,489,6,10,2018-2019,0,Faible,Moins de 10 matchs de C1; Historique C1 ancien : dernière saison 2018-2019
1,Arsenal,733,78,4,2025-2026,38,Élevée,Au moins 30 matchs de C1 et présence depuis 2023-2024
2,Aston Villa,489,12,0,2024-2025,12,Modérée,Moins de 30 matchs de C1
3,Atlético de Madrid,769,131,0,2025-2026,36,Élevée,Au moins 30 matchs de C1 et présence depuis 2023-2024
4,FC Barcelona,800,142,0,2025-2026,36,Élevée,Au moins 30 matchs de C1 et présence depuis 2023-2024
5,Bayern München,724,160,0,2025-2026,40,Élevée,Au moins 30 matchs de C1 et présence depuis 2023-2024
6,Bodø/Glimt,130,12,18,2025-2026,12,Modérée,Moins de 30 matchs de C1
7,Borussia Dortmund,693,124,0,2025-2026,37,Élevée,Au moins 30 matchs de C1 et présence depuis 2023-2024
8,Club Brugge,582,60,16,2025-2026,22,Élevée,Au moins 30 matchs de C1 et présence depuis 2023-2024
9,Como,93,0,0,<NA>,0,Très faible,Aucun match CL dans le fichier


## 4. Les six clubs sans C1 observée et la règle de traitement

Les six noms attendus sont **Como, Fenerbahçe, LASK, Real Betis, Sabah et Viking**.
L’assertion suivante vérifie ce résultat dans le fichier local.

**Absence dans l’échantillon ≠ absence dans l’histoire du club.** La fenêtre C1 du
fichier commence en 2012-2013. Les anciens parcours de Fenerbahçe ou du Betis ne sont
pas couverts. Un match de `CLQ` n’est pas un match de la phase principale.
À l’inverse, des centaines de matchs nationaux ne remplacent pas des observations de C1.
Sabah cumule deux limites : aucune C1 et très peu de rencontres toutes compétitions,
car la couverture de son championnat est insuffisante dans cette base.

**Règle proposée pour le modèle :** conserver les 36 clubs, ajouter les indicateurs
`sans_historique_c1` et `nb_matchs_c1_recents`, et utiliser les informations de niveau
disponibles avant le match (Elo, forme nationale, effectif). Un Elo manquant reste
signalé comme manquant ; ce notebook ne crée pas de faux Elo.

Pour chaque moyenne propre à la C1, utiliser un lissage vers la moyenne C1 de
l’échantillon d’entraînement :

$$
w = \frac{n_{\mathrm{C1,récents}}}{n_{\mathrm{C1,récents}} + 20},
\qquad x_{\mathrm{lissé}} = w x_{\mathrm{club,récents}} + (1-w)\mu_{\mathrm{C1,train}}.
$$

Les statistiques du club et le compteur utilisent la **même fenêtre de trois saisons**.
Si aucun match récent n’est observé, `w = 0` et on utilise la moyenne d’entraînement,
avec un indicateur d’absence. Cela couvre aussi les retours après une longue absence
(par exemple Roma), sans assimiler une donnée manquante à une performance nulle.
Pour une moyenne comportant des valeurs manquantes, `n` doit compter uniquement les
observations utilisables de cette moyenne, pas tous les matchs du club.

Le paramètre `20`, la fenêtre et les seuils sont à choisir sur les saisons de validation,
jamais sur la saison prédite. La moyenne de référence et les statistiques doivent être
recalculées uniquement avec les matchs antérieurs à chaque match lors d’un backtest.
L’incertitude doit être signalée et la calibration évaluée séparément pour les petits historiques.

In [5]:
SANS_C1_ATTENDUS = {"Como", "Fenerbahçe", "LASK", "Real Betis", "Sabah", "Viking"}
sans_c1 = participants.loc[participants.sans_historique_c1].copy()
assert set(sans_c1.nom_uefa) == SANS_C1_ATTENDUS, "Le fichier a changé : revoir la liste et les conclusions."

LISSAGE_K = 20
participants["poids_historique_c1_recent"] = (
    participants.nb_matchs_c1_recents / (participants.nb_matchs_c1_recents + LISSAGE_K)
)
participants["regle_traitement"] = participants.nb_matchs_c1_recents.map(
    lambda n: "Prior C1 entraînement + indicateur absence ; niveau national si disponible"
    if n == 0 else "Moyenne C1 récente lissée vers le prior entraînement ; niveau national si disponible"
)

def lisser_stat_c1(estimation_club, nb_observations, moyenne_entrainement, k=LISSAGE_K):
    """Lisser une moyenne C1 ; n compte les valeurs réellement utilisables avant le match."""
    if pd.isna(nb_observations) or nb_observations < 0 or k <= 0:
        raise ValueError("Le nombre d’observations doit être positif ou nul et k strictement positif.")
    if pd.isna(moyenne_entrainement):
        raise ValueError("La moyenne d’entraînement doit être disponible.")
    if nb_observations == 0:
        return moyenne_entrainement
    if pd.isna(estimation_club):
        raise ValueError("Une estimation est requise lorsque n > 0 ; vérifier les valeurs manquantes.")
    poids = nb_observations / (nb_observations + k)
    return poids * estimation_club + (1 - poids) * moyenne_entrainement

display(participants.loc[participants.sans_historique_c1,
    ["nom_uefa", "nb_matchs_games", "nb_matchs_c1", "nb_matchs_clq", "fiabilite", "regle_traitement"]])
print("Exemple pédagogique, valeurs fictives :", lisser_stat_c1(0.7, 10, 0.5))

,nom_uefa,nb_matchs_games,nb_matchs_c1,nb_matchs_clq,fiabilite,regle_traitement
9,Como,93,0,0,Très faible,Prior C1 entraînement + indicateur absence ; n...
10,Fenerbahçe,626,0,24,Très faible,Prior C1 entraînement + indicateur absence ; n...
14,LASK,114,0,4,Très faible,Prior C1 entraînement + indicateur absence ; n...
25,Real Betis,630,0,0,Très faible,Prior C1 entraînement + indicateur absence ; n...
28,Sabah,14,0,0,Très faible,Prior C1 entraînement + indicateur absence ; n...
34,Viking,51,0,0,Très faible,Prior C1 entraînement + indicateur absence ; n...


Exemple pédagogique, valeurs fictives : 0.5666666666666667


## 5. Calendrier de la phase de ligue 2026-2027

La source par journée fixe les dates du 8 septembre 2026 au 27 janvier 2027.
La liste indépendante du tirage contrôle les adversaires et le sens domicile/extérieur.
Le premier dictionnaire contient **une seule entrée par rencontre**, sans score.
Les horaires ne sont pas repris : ils ne sont pas nécessaires à la simulation demandée.
`match_id` est une clé locale déterministe, pas l’identifiant UEFA ni celui de `games.csv`.

In [6]:
CALENDRIER_UEFA_2026 = {1: {'2026-09-08': [('AEK Athens', 'LASK'),
                    ('Borussia Dortmund', 'Villarreal'),
                    ('Club Brugge', 'Aston Villa'),
                    ('Lille', 'Real Betis'),
                    ('Porto', 'Man City'),
                    ('Real Madrid', 'Inter')],
     '2026-09-09': [('FC Barcelona', 'Feyenoord'),
                    ('Liverpool', 'Atlético de Madrid'),
                    ('Napoli', 'Arsenal'),
                    ('Paris', 'Slovan Bratislava'),
                    ('Sporting CP', 'Galatasaray'),
                    ('Stuttgart', 'Viking')],
     '2026-09-10': [('Bayern München', 'Bodø/Glimt'),
                    ('Como', 'Leipzig'),
                    ('Fenerbahçe', 'Roma'),
                    ('Man United', 'Sabah'),
                    ('PSV', 'Shakhtar'),
                    ('Slavia Praha', 'Lens')]},
 2: {'2026-10-13': [('Arsenal', 'Lille'),
                    ('Atlético de Madrid', 'Man United'),
                    ('Galatasaray', 'FC Barcelona'),
                    ('Inter', 'Club Brugge'),
                    ('Leipzig', 'PSV'),
                    ('Lens', 'Sporting CP'),
                    ('Sabah', 'Slavia Praha'),
                    ('Viking', 'Bayern München'),
                    ('Villarreal', 'Napoli')],
     '2026-10-14': [('Aston Villa', 'Fenerbahçe'),
                    ('Bodø/Glimt', 'Borussia Dortmund'),
                    ('Feyenoord', 'Como'),
                    ('LASK', 'Liverpool'),
                    ('Man City', 'Paris'),
                    ('Real Betis', 'Porto'),
                    ('Roma', 'Real Madrid'),
                    ('Shakhtar', 'AEK Athens'),
                    ('Slovan Bratislava', 'Stuttgart')]},
 3: {'2026-10-20': [('Fenerbahçe', 'Slavia Praha'),
                    ('Liverpool', 'Villarreal'),
                    ('Man City', 'AEK Athens'),
                    ('Napoli', 'Bodø/Glimt'),
                    ('Paris', 'FC Barcelona'),
                    ('Porto', 'PSV'),
                    ('Roma', 'Slovan Bratislava'),
                    ('Sabah', 'Borussia Dortmund'),
                    ('Stuttgart', 'Atlético de Madrid')],
     '2026-10-21': [('Aston Villa', 'Viking'),
                    ('Bayern München', 'Arsenal'),
                    ('Club Brugge', 'Lens'),
                    ('Como', 'Man United'),
                    ('Inter', 'Shakhtar'),
                    ('Lille', 'Galatasaray'),
                    ('Real Betis', 'Feyenoord'),
                    ('Real Madrid', 'Leipzig'),
                    ('Sporting CP', 'LASK')]},
 4: {'2026-11-03': [('Atlético de Madrid', 'Bayern München'),
                    ('Bodø/Glimt', 'Lille'),
                    ('FC Barcelona', 'Aston Villa'),
                    ('Feyenoord', 'Inter'),
                    ('Galatasaray', 'Stuttgart'),
                    ('LASK', 'Slovan Bratislava'),
                    ('Man United', 'Roma'),
                    ('Shakhtar', 'Sporting CP'),
                    ('Villarreal', 'Paris')],
     '2026-11-04': [('AEK Athens', 'Real Madrid'),
                    ('Borussia Dortmund', 'Real Betis'),
                    ('Fenerbahçe', 'Liverpool'),
                    ('Leipzig', 'Man City'),
                    ('Lens', 'Como'),
                    ('PSV', 'Club Brugge'),
                    ('Porto', 'Napoli'),
                    ('Slavia Praha', 'Arsenal'),
                    ('Viking', 'Sabah')]},
 5: {'2026-11-24': [('Arsenal', 'Borussia Dortmund'),
                    ('Bodø/Glimt', 'LASK'),
                    ('Como', 'AEK Athens'),
                    ('Feyenoord', 'Porto'),
                    ('Galatasaray', 'Aston Villa'),
                    ('Leipzig', 'Lens'),
                    ('Man City', 'Napoli'),
                    ('Real Madrid', 'PSV'),
                    ('Slovan Bratislava', 'Real Betis')],
     '2026-11-25': [('Atlético de Madrid', 'Viking'),
                    ('Club Brugge', 'Liverpool'),
                    ('Inter', 'Stuttgart'),
                    ('Lille', 'Bayern München'),
                    ('Paris', 'Roma'),
                    ('Sabah', 'FC Barcelona'),
                    ('Shakhtar', 'Fenerbahçe'),
                    ('Slavia Praha', 'Villarreal'),
                    ('Sporting CP', 'Man United')]},
 6: {'2026-12-08': [('AEK Athens', 'Galatasaray'),
                    ('Aston Villa', 'Paris'),
                    ('Bayern München', 'Slavia Praha'),
                    ('FC Barcelona', 'Man City'),
                    ('Man United', 'Leipzig'),
                    ('Napoli', 'Club Brugge'),
                    ('Roma', 'Sporting CP'),
                    ('Viking', 'Feyenoord'),
                    ('Villarreal', 'Sabah')],
     '2026-12-09': [('Arsenal', 'Real Madrid'),
                    ('Borussia Dortmund', 'Inter'),
                    ('LASK', 'Fenerbahçe'),
                    ('Lens', 'Bodø/Glimt'),
                    ('Liverpool', 'Porto'),
                    ('PSV', 'Atlético de Madrid'),
                    ('Real Betis', 'Como'),
                    ('Slovan Bratislava', 'Shakhtar'),
                    ('Stuttgart', 'Lille')]},
 7: {'2027-01-19': [('AEK Athens', 'Roma'),
                    ('Aston Villa', 'Borussia Dortmund'),
                    ('Bodø/Glimt', 'Atlético de Madrid'),
                    ('Galatasaray', 'Feyenoord'),
                    ('Inter', 'Liverpool'),
                    ('Lille', 'Slovan Bratislava'),
                    ('Porto', 'Slavia Praha'),
                    ('Real Madrid', 'LASK'),
                    ('Stuttgart', 'Club Brugge')],
     '2027-01-20': [('Como', 'Paris'),
                    ('Fenerbahçe', 'Villarreal'),
                    ('Leipzig', 'Shakhtar'),
                    ('Lens', 'Man City'),
                    ('Man United', 'Bayern München'),
                    ('Real Betis', 'Arsenal'),
                    ('Sabah', 'Napoli'),
                    ('Sporting CP', 'FC Barcelona'),
                    ('Viking', 'PSV')]},
 8: {'2027-01-27': [('Arsenal', 'Sabah'),
                    ('Atlético de Madrid', 'Fenerbahçe'),
                    ('Bayern München', 'Real Betis'),
                    ('Borussia Dortmund', 'AEK Athens'),
                    ('Club Brugge', 'Bodø/Glimt'),
                    ('FC Barcelona', 'Como'),
                    ('Feyenoord', 'Leipzig'),
                    ('LASK', 'Porto'),
                    ('Liverpool', 'Lens'),
                    ('Man City', 'Sporting CP'),
                    ('Napoli', 'Viking'),
                    ('PSV', 'Stuttgart'),
                    ('Paris', 'Galatasaray'),
                    ('Roma', 'Lille'),
                    ('Shakhtar', 'Real Madrid'),
                    ('Slavia Praha', 'Aston Villa'),
                    ('Slovan Bratislava', 'Inter'),
                    ('Villarreal', 'Man United')]}}

In [7]:
# Liste de contrôle indépendante, transcrite depuis la page UEFA du tirage.
RIVAUX_UEFA_2026 = {'AEK Athens': {'domicile': ['Real Madrid', 'Roma', 'Galatasaray', 'LASK'],
                'exterieur': ['Man City', 'Borussia Dortmund', 'Shakhtar', 'Como']},
 'Arsenal': {'domicile': ['Real Madrid', 'Borussia Dortmund', 'Lille', 'Sabah'],
             'exterieur': ['Bayern München', 'Real Betis', 'Napoli', 'Slavia Praha']},
 'Aston Villa': {'domicile': ['Paris', 'Borussia Dortmund', 'Fenerbahçe', 'Viking'],
                 'exterieur': ['FC Barcelona', 'Club Brugge', 'Galatasaray', 'Slavia Praha']},
 'Atlético de Madrid': {'domicile': ['Bayern München', 'Man United', 'Fenerbahçe', 'Viking'],
                        'exterieur': ['Liverpool', 'PSV', 'Bodø/Glimt', 'Stuttgart']},
 'Borussia Dortmund': {'domicile': ['Inter', 'Real Betis', 'Villarreal', 'AEK Athens'],
                       'exterieur': ['Arsenal', 'Aston Villa', 'Bodø/Glimt', 'Sabah']},
 'FC Barcelona': {'domicile': ['Man City', 'Aston Villa', 'Feyenoord', 'Como'],
                  'exterieur': ['Paris', 'Sporting CP', 'Galatasaray', 'Sabah']},
 'Bayern München': {'domicile': ['Arsenal', 'Real Betis', 'Bodø/Glimt', 'Slavia Praha'],
                    'exterieur': ['Atlético de Madrid', 'Man United', 'Lille', 'Viking']},
 'Bodø/Glimt': {'domicile': ['Atlético de Madrid', 'Borussia Dortmund', 'Lille', 'LASK'],
                'exterieur': ['Bayern München', 'Club Brugge', 'Napoli', 'Lens']},
 'Club Brugge': {'domicile': ['Liverpool', 'Aston Villa', 'Bodø/Glimt', 'Lens'],
                 'exterieur': ['Inter', 'PSV', 'Napoli', 'Stuttgart']},
 'Como': {'domicile': ['Paris', 'Man United', 'Leipzig', 'AEK Athens'],
          'exterieur': ['FC Barcelona', 'Real Betis', 'Feyenoord', 'Lens']},
 'Fenerbahçe': {'domicile': ['Liverpool', 'Roma', 'Villarreal', 'Slavia Praha'],
                'exterieur': ['Atlético de Madrid', 'Aston Villa', 'Shakhtar', 'LASK']},
 'Feyenoord': {'domicile': ['Inter', 'Porto', 'Leipzig', 'Como'],
               'exterieur': ['FC Barcelona', 'Real Betis', 'Galatasaray', 'Viking']},
 'Galatasaray': {'domicile': ['FC Barcelona', 'Aston Villa', 'Feyenoord', 'Stuttgart'],
                 'exterieur': ['Paris', 'Sporting CP', 'Lille', 'AEK Athens']},
 'Inter': {'domicile': ['Liverpool', 'Club Brugge', 'Shakhtar', 'Stuttgart'],
           'exterieur': ['Real Madrid', 'Borussia Dortmund', 'Feyenoord', 'Slovan Bratislava']},
 'LASK': {'domicile': ['Liverpool', 'Porto', 'Fenerbahçe', 'Slovan Bratislava'],
          'exterieur': ['Real Madrid', 'Sporting CP', 'Bodø/Glimt', 'AEK Athens']},
 'Leipzig': {'domicile': ['Man City', 'PSV', 'Shakhtar', 'Lens'],
             'exterieur': ['Real Madrid', 'Man United', 'Feyenoord', 'Como']},
 'Lens': {'domicile': ['Man City', 'Sporting CP', 'Bodø/Glimt', 'Como'],
          'exterieur': ['Liverpool', 'Club Brugge', 'Leipzig', 'Slavia Praha']},
 'Lille': {'domicile': ['Bayern München', 'Real Betis', 'Galatasaray', 'Slovan Bratislava'],
           'exterieur': ['Arsenal', 'Roma', 'Bodø/Glimt', 'Stuttgart']},
 'Liverpool': {'domicile': ['Atlético de Madrid', 'Porto', 'Villarreal', 'Lens'],
               'exterieur': ['Inter', 'Club Brugge', 'Fenerbahçe', 'LASK']},
 'Man City': {'domicile': ['Paris', 'Sporting CP', 'Napoli', 'AEK Athens'],
              'exterieur': ['FC Barcelona', 'Porto', 'Leipzig', 'Lens']},
 'Man United': {'domicile': ['Bayern München', 'Roma', 'Leipzig', 'Sabah'],
                'exterieur': ['Atlético de Madrid', 'Sporting CP', 'Villarreal', 'Como']},
 'Napoli': {'domicile': ['Arsenal', 'Club Brugge', 'Bodø/Glimt', 'Viking'],
            'exterieur': ['Man City', 'Porto', 'Villarreal', 'Sabah']},
 'Paris': {'domicile': ['FC Barcelona', 'Roma', 'Galatasaray', 'Slovan Bratislava'],
           'exterieur': ['Man City', 'Aston Villa', 'Villarreal', 'Como']},
 'Porto': {'domicile': ['Man City', 'PSV', 'Napoli', 'Slavia Praha'],
           'exterieur': ['Liverpool', 'Real Betis', 'Feyenoord', 'LASK']},
 'PSV': {'domicile': ['Atlético de Madrid', 'Club Brugge', 'Shakhtar', 'Stuttgart'],
         'exterieur': ['Real Madrid', 'Porto', 'Leipzig', 'Viking']},
 'Real Betis': {'domicile': ['Arsenal', 'Porto', 'Feyenoord', 'Como'],
                'exterieur': ['Bayern München', 'Borussia Dortmund', 'Lille', 'Slovan Bratislava']},
 'Real Madrid': {'domicile': ['Inter', 'PSV', 'Leipzig', 'LASK'],
                 'exterieur': ['Arsenal', 'Roma', 'Shakhtar', 'AEK Athens']},
 'Roma': {'domicile': ['Real Madrid', 'Sporting CP', 'Lille', 'Slovan Bratislava'],
          'exterieur': ['Paris', 'Man United', 'Fenerbahçe', 'AEK Athens']},
 'Sabah': {'domicile': ['FC Barcelona', 'Borussia Dortmund', 'Napoli', 'Slavia Praha'],
           'exterieur': ['Arsenal', 'Man United', 'Villarreal', 'Viking']},
 'Shakhtar': {'domicile': ['Real Madrid', 'Sporting CP', 'Fenerbahçe', 'AEK Athens'],
              'exterieur': ['Inter', 'PSV', 'Leipzig', 'Slovan Bratislava']},
 'Slavia Praha': {'domicile': ['Arsenal', 'Aston Villa', 'Villarreal', 'Lens'],
                  'exterieur': ['Bayern München', 'Porto', 'Fenerbahçe', 'Sabah']},
 'Slovan Bratislava': {'domicile': ['Inter', 'Real Betis', 'Shakhtar', 'Stuttgart'],
                       'exterieur': ['Paris', 'Roma', 'Lille', 'LASK']},
 'Sporting CP': {'domicile': ['FC Barcelona', 'Man United', 'Galatasaray', 'LASK'],
                 'exterieur': ['Man City', 'Roma', 'Shakhtar', 'Lens']},
 'Stuttgart': {'domicile': ['Atlético de Madrid', 'Club Brugge', 'Lille', 'Viking'],
               'exterieur': ['Inter', 'PSV', 'Galatasaray', 'Slovan Bratislava']},
 'Viking': {'domicile': ['Bayern München', 'PSV', 'Feyenoord', 'Sabah'],
            'exterieur': ['Atlético de Madrid', 'Aston Villa', 'Napoli', 'Stuttgart']},
 'Villarreal': {'domicile': ['Paris', 'Man United', 'Napoli', 'Sabah'],
                'exterieur': ['Liverpool', 'Borussia Dortmund', 'Fenerbahçe', 'Slavia Praha']}}

In [8]:
rencontres = []
for journee, dates in CALENDRIER_UEFA_2026.items():
    numero = 0
    for date, affiches in sorted(dates.items()):
        for domicile, exterieur in sorted(affiches):
            numero += 1
            rencontres.append({
                "match_id": f"2026-L{journee:02d}-{numero:02d}", "saison": SAISON_CIBLE,
                "journee": journee, "date": date,
                "domicile_uefa": domicile, "exterieur_uefa": exterieur,
                "domicile_club_id": PARTICIPANTS_2026[domicile]["club_id"],
                "exterieur_club_id": PARTICIPANTS_2026[exterieur]["club_id"],
            })
matchs = pd.DataFrame(rencontres).sort_values(["journee", "date", "match_id"]).reset_index(drop=True)
matchs["date"] = pd.to_datetime(matchs["date"], errors="raise")

# Deux perspectives par rencontre : le fichier demandé donne les huit lignes de chaque club.
lignes = []
for m in matchs.to_dict("records"):
    for cote, autre, lieu in [("domicile", "exterieur", "domicile"),
                              ("exterieur", "domicile", "exterieur")]:
        lignes.append({
            "match_id": m["match_id"], "saison": m["saison"], "journee": m["journee"], "date": m["date"],
            "club_id": m[f"{cote}_club_id"], "nom_uefa": m[f"{cote}_uefa"],
            "adversaire_club_id": m[f"{autre}_club_id"], "adversaire_uefa": m[f"{autre}_uefa"],
            "lieu": lieu, "domicile_club_id": m["domicile_club_id"], "exterieur_club_id": m["exterieur_club_id"],
        })
calendrier = pd.DataFrame(lignes).sort_values(["nom_uefa", "journee"]).reset_index(drop=True)
calendrier["source_uefa"] = SOURCE_CALENDRIER
calendrier["date_verification_uefa"] = DATE_VERIFICATION_UEFA

def cellule_adversaire(m):
    return f"{m.adversaire_uefa} ({'D' if m.lieu == 'domicile' else 'E'}) — {m.date:%d/%m/%Y}"

calendrier_vue = calendrier.assign(adversaire=[cellule_adversaire(m) for m in calendrier.itertuples(index=False)])
calendrier_vue = calendrier_vue.pivot(index="nom_uefa", columns="journee", values="adversaire")
calendrier_vue.columns = [f"J{j}" for j in calendrier_vue.columns]
print("D = domicile ; E = extérieur. Export : une ligne par club et par journée.")
with pd.option_context("display.max_rows", 40, "display.max_columns", 8, "display.max_colwidth", 55):
    display(calendrier_vue)

D = domicile ; E = extérieur. Export : une ligne par club et par journée.


,J1,J2,J3,J4,J5,J6,J7,J8
nom_uefa,,,,,,,,
AEK Athens,LASK (D) — 08/09/2026,Shakhtar (E) — 14/10/2026,Man City (E) — 20/10/2026,Real Madrid (D) — 04/11/2026,Como (E) — 24/11/2026,Galatasaray (D) — 08/12/2026,Roma (D) — 19/01/2027,Borussia Dortmund (E) — 27/01/2027
Arsenal,Napoli (E) — 09/09/2026,Lille (D) — 13/10/2026,Bayern München (E) — 21/10/2026,Slavia Praha (E) — 04/11/2026,Borussia Dortmund (D) — 24/11/2026,Real Madrid (D) — 09/12/2026,Real Betis (E) — 20/01/2027,Sabah (D) — 27/01/2027
Aston Villa,Club Brugge (E) — 08/09/2026,Fenerbahçe (D) — 14/10/2026,Viking (D) — 21/10/2026,FC Barcelona (E) — 03/11/2026,Galatasaray (E) — 24/11/2026,Paris (D) — 08/12/2026,Borussia Dortmund (D) — 19/01/2027,Slavia Praha (E) — 27/01/2027
Atlético de Madrid,Liverpool (E) — 09/09/2026,Man United (D) — 13/10/2026,Stuttgart (E) — 20/10/2026,Bayern München (D) — 03/11/2026,Viking (D) — 25/11/2026,PSV (E) — 09/12/2026,Bodø/Glimt (E) — 19/01/2027,Fenerbahçe (D) — 27/01/2027
Bayern München,Bodø/Glimt (D) — 10/09/2026,Viking (E) — 13/10/2026,Arsenal (D) — 21/10/2026,Atlético de Madrid (E) — 03/11/2026,Lille (E) — 25/11/2026,Slavia Praha (D) — 08/12/2026,Man United (E) — 20/01/2027,Real Betis (D) — 27/01/2027
Bodø/Glimt,Bayern München (E) — 10/09/2026,Borussia Dortmund (D) — 14/10/2026,Napoli (E) — 20/10/2026,Lille (D) — 03/11/2026,LASK (D) — 24/11/2026,Lens (E) — 09/12/2026,Atlético de Madrid (D) — 19/01/2027,Club Brugge (E) — 27/01/2027
Borussia Dortmund,Villarreal (D) — 08/09/2026,Bodø/Glimt (E) — 14/10/2026,Sabah (E) — 20/10/2026,Real Betis (D) — 04/11/2026,Arsenal (E) — 24/11/2026,Inter (D) — 09/12/2026,Aston Villa (E) — 19/01/2027,AEK Athens (D) — 27/01/2027
Club Brugge,Aston Villa (D) — 08/09/2026,Inter (E) — 13/10/2026,Lens (D) — 21/10/2026,PSV (E) — 04/11/2026,Liverpool (D) — 25/11/2026,Napoli (E) — 08/12/2026,Stuttgart (E) — 19/01/2027,Bodø/Glimt (D) — 27/01/2027
Como,Leipzig (D) — 10/09/2026,Feyenoord (E) — 14/10/2026,Man United (D) — 21/10/2026,Lens (E) — 04/11/2026,AEK Athens (D) — 24/11/2026,Real Betis (E) — 09/12/2026,Paris (D) — 20/01/2027,FC Barcelona (E) — 27/01/2027


## 6. Contrôles et export des livrables

Avant d’écrire les fichiers, on contrôle 36 clubs, 144 rencontres, 18 matchs par
journée, huit adversaires distincts par club, quatre réceptions et quatre déplacements.
On vérifie aussi que chaque club joue une fois par journée, que les deux perspectives
d’un match sont cohérentes et que les 288 adversaires concordent avec le tirage UEFA.

In [9]:
assert len(matchs) == 144 and matchs.match_id.is_unique
assert not matchs.duplicated(["domicile_club_id", "exterieur_club_id"]).any()
assert matchs.groupby("journee").size().to_dict() == {j: 18 for j in range(1, 9)}
assert len(calendrier) == 288
assert set(calendrier.club_id) == set(participants.club_id)
assert calendrier.groupby("club_id").size().eq(8).all()
assert calendrier.groupby("club_id").adversaire_club_id.nunique().eq(8).all()
assert calendrier.groupby(["club_id", "journee"]).size().eq(1).all()
assert calendrier.groupby(["club_id", "lieu"]).size().eq(4).all()
assert calendrier.groupby("match_id").size().eq(2).all()
assert calendrier.club_id.ne(calendrier.adversaire_club_id).all()
assert not calendrier[["club_id", "adversaire_club_id", "date"]].isna().any().any()
assert calendrier.date.min() == DATE_DEBUT_PHASE_LIGUE
assert calendrier.date.max() == pd.Timestamp("2027-01-27")
assert set(RIVAUX_UEFA_2026) == set(PARTICIPANTS_2026)
for club, rivaux in RIVAUX_UEFA_2026.items():
    for lieu in ("domicile", "exterieur"):
        observes = calendrier.loc[calendrier.nom_uefa.eq(club) & calendrier.lieu.eq(lieu), "adversaire_uefa"]
        assert len(rivaux[lieu]) == 4 and set(observes) == set(rivaux[lieu]), (club, lieu)
for match_id, perspectives in calendrier.groupby("match_id"):
    d = perspectives.loc[perspectives.lieu.eq("domicile")].iloc[0]
    e = perspectives.loc[perspectives.lieu.eq("exterieur")].iloc[0]
    assert d.club_id == e.adversaire_club_id and e.club_id == d.adversaire_club_id
    assert d.date == e.date and d.journee == e.journee

assert participants.nb_matchs_c1.le(participants.nb_matchs_games).all()
assert participants.nb_matchs_c1_recents.le(participants.nb_matchs_c1).all()
assert participants.loc[participants.sans_historique_c1, "derniere_saison_c1"].isna().all()
assert participants.loc[participants.sans_historique_c1, "poids_historique_c1_recent"].eq(0).all()

participants["source_uefa"] = SOURCE_PARTICIPANTS
participants["date_verification_uefa"] = DATE_VERIFICATION_UEFA
participants["date_debut_phase_ligue"] = DATE_DEBUT_PHASE_LIGUE.strftime("%Y-%m-%d")
participants["premiere_saison_c1_observee"] = int(ucl.season.min())
participants["derniere_saison_c1_observee"] = int(ucl.season.max())
participants["derniere_date_c1_observee"] = ucl.date.max().strftime("%Y-%m-%d")
participants["premiere_saison_fenetre_recente"] = PREMIERE_SAISON_RECENTE
participants["lissage_k"] = LISSAGE_K
OUTPUT.mkdir(parents=True, exist_ok=True)
fichier_participants = OUTPUT / "participants_2026.csv"
fichier_calendrier = OUTPUT / "calendrier_2026.csv"
participants.to_csv(fichier_participants, index=False, encoding="utf-8-sig", na_rep="")
calendrier.to_csv(fichier_calendrier, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")

# Relire les exports vérifie le schéma, les accents, les identifiants et les volumes.
verif_p = pd.read_csv(fichier_participants)
verif_c = pd.read_csv(fichier_calendrier, parse_dates=["date"])
assert len(verif_p) == 36 and verif_p.club_id.is_unique
assert len(verif_c) == 288 and verif_c.match_id.nunique() == 144
assert set(verif_c.club_id) == set(verif_p.club_id)
assert set(verif_p.nom_uefa) == set(PARTICIPANTS_2026)
print("Contrôles réussis : 36 clubs, 144 rencontres, 288 perspectives, 8 adversaires et 4 D / 4 E chacun.")
print(f"Exports créés : {fichier_participants} et {fichier_calendrier}")
display(participants.groupby("fiabilite").size().rename("nombre_de_clubs").to_frame())

Contrôles réussis : 36 clubs, 144 rencontres, 288 perspectives, 8 adversaires et 4 D / 4 E chacun.
Exports créés : C:\Users\Mouss\OneDrive\Desktop\CHAMPIONS-\output\participants_2026.csv et C:\Users\Mouss\OneDrive\Desktop\CHAMPIONS-\output\calendrier_2026.csv


,nombre_de_clubs
fiabilite,
Faible,5
Modérée,5
Très faible,6
Élevée,20


## 7. Ce que Maxime doit savoir expliquer

« Un club est peu fiable pour notre modèle quand on a peu d’observations de C1 ou
qu’elles sont anciennes. Cela ne signifie pas qu’il est faible. Como, Fenerbahçe,
LASK, Betis, Sabah et Viking n’ont aucun match `CL` dans notre fichier avant la saison
cible. Nous les conservons : nous signalons l’absence d’historique et utilisons les
informations nationales disponibles. Les moyennes de C1 sont ramenées vers une
moyenne d’entraînement, avec un poids qui augmente quand le nombre de matchs
récents augmente. Les seuils et le lissage devront être validés sur des saisons passées. »

Exemples à commenter : AEK et Roma ont un historique de C1 ancien ; Lens n’a que six
matchs observés ; Sabah possède aussi très peu de matchs toutes compétitions.
Un petit nombre de matchs rend une moyenne plus sensible au tirage et aux résultats extrêmes.

**Limites :** le fichier ne couvre pas toute l’histoire des clubs ni tous les championnats.
La saison 2025-2026 de C1 est incomplète (la finale manque dans le fichier actuel).
Les comptes sont donc des volumes observés. Ce notebook n’entraîne pas le modèle,
ne corrige pas les huit scores avec tirs au but de l’EDA et ne calcule pas de statistiques
de buts à partir de ces scores. Lors de la modélisation, réutiliser les scores nettoyés
et recalculer les indicateurs avant chaque match.

Le calendrier est une photographie des annonces UEFA à la date de consultation ; les dates
peuvent être modifiées par l’UEFA. Pour mettre à jour les livrables, modifier les dictionnaires
après vérification de la source puis relancer tout le notebook. `output/` et les CSV
restent ignorés par Git, conformément à la configuration du dépôt ; le notebook contient
toutes les informations nécessaires à leur reconstruction.